Notebook to read and explore BRW file using Herding Spikes

In [ ]:
# path to the HS2 code, for imports
hs2_dir = 'C:\\Users\\Admin\\OneDrive\\Desktop\\UniBE\\src\\HS2\\'

from datetime import datetime
import sys, os

sys.path.append(hs2_dir)

# from herdingspikes.hs2 import HSDetection
# from herdingspikes.hs2 import HSClustering
# from herdingspikes.probe import BioCam

import matplotlib.pyplot as plt
import numpy as np
import pylab
import h5py
import json

In [ ]:
# PARAMETERS
##########################

# raw data file
data_file = "C:\\Users\\Admin\\OneDrive\\Desktop\\UniBE\\data\\150525_530_15_1a_DRC470_02.brw"
out_file = data_file.replace('.brw','')

# detection parameters
##########################

to_localize = True

# Spike cut-out size (values below are for 7kHz)
cutout_start = 4
cutout_end = 14
cutout_length = cutout_end+cutout_start+1

# detection threshold
threshold = 16


In [7]:
with h5py.File(data_file, 'r') as f:
    for key in f:
        print(f"Group: {key}")
        for subkey in f[key]:
            print(f"  Subgroup: {subkey}")

Group: ExperimentSettings
  Subgroup: b'{\r\n  "TimelineEvents": [],\r\n  "ValueConverter": {\r\n    "MinAnalogValue": -8000.0,\r\n    "MaxAnalogValue": 8000.0,\r\n    "MaxDigitalValue": 4095,\r\n    "MinDigitalValue": 0,\r\n    "ScaleFactor": 1.0,\r\n    "JsonVersion": 0\r\n  },\r\n  "TimeConverter": {\r\n    "FrameRate": 10103.0751953125\r\n  },\r\n  "ExperimentType": "Standard",\r\n  "ExperimentSettings": {\r\n    "Description": null,\r\n    "CalibrationCh": {\r\n      "Well": 1,\r\n      "Row": 1,\r\n      "Col": 1\r\n    },\r\n    "JsonVersion": 0\r\n  },\r\n  "SourceFilterSettings": {\r\n    "HighPassFilterCutOffFrequency": 100.0,\r\n    "HighPassFilterOrder": 2,\r\n    "JsonVersion": 0\r\n  },\r\n  "BiologicalModel": "Retina",\r\n  "TimelineIntervals": [],\r\n  "DataSettings": {\r\n    "Noise": {\r\n      "$type": "NoiseInfoDetectionSettings",\r\n      "DiscardOutliers": true,\r\n      "JsonVersion": 0\r\n    },\r\n    "Spikes": {\r\n      "$type": "SpikeDetectionSettings",\r\n 

In [4]:
with h5py.File(data_file, 'r') as f:
    sparse = f['Well_A1']['EventsBasedSparseRaw'][:]
    metadata = f['ExperimentSettings'][:]

json_str = metadata[0].decode('utf-8') # python dict serialized into byte format
my_dict = json.loads(json_str)
print(my_dict)
MaxVolt = my_dict['ValueConverter']['MaxAnalogValue']   # mV
MinVolt = my_dict['ValueConverter']['MinAnalogValue']  # mV
BitDepth = 16
SignalInversion = 1
MVOffset = SignalInversion * MinVolt   # Assume 0 unless otherwise specified

{'TimelineEvents': [], 'ValueConverter': {'MinAnalogValue': -8000.0, 'MaxAnalogValue': 8000.0, 'MaxDigitalValue': 4095, 'MinDigitalValue': 0, 'ScaleFactor': 1.0, 'JsonVersion': 0}, 'TimeConverter': {'FrameRate': 10103.0751953125}, 'ExperimentType': 'Standard', 'ExperimentSettings': {'Description': None, 'CalibrationCh': {'Well': 1, 'Row': 1, 'Col': 1}, 'JsonVersion': 0}, 'SourceFilterSettings': {'HighPassFilterCutOffFrequency': 100.0, 'HighPassFilterOrder': 2, 'JsonVersion': 0}, 'BiologicalModel': 'Retina', 'TimelineIntervals': [], 'DataSettings': {'Noise': {'$type': 'NoiseInfoDetectionSettings', 'DiscardOutliers': True, 'JsonVersion': 0}, 'EventsBasedRawRanges': {'$type': 'NoiseBlankingCompressionSettings', 'HigherPositiveStdFactor': 4.0, 'LowerPositiveStdFactor': 3.5, 'HigherNegativeStdFactor': 4.0, 'LowerNegativeStdFactor': 3.5, 'PreEventExtraSavedLengthMs': 0.0, 'PostEventExtraSavedLengthMs': 0.0, 'NeighborsScope': 'NoNeighbors', 'JsonVersion': 0}}, 'MeaSystem': {'HwVersion': {'Maj

In [5]:
print(len(sparse))

3096086286
